In [13]:
import pandas as pd
from pandas import read_csv

df = pd.read_csv("202609_RecebimentosRecursosPorFavorecido.csv", delimiter=";", encoding="latin-1")

df.head(2)

,Código Favorecido,Nome Favorecido,Sigla UF,Nome Município,Código Órgão Superior,Nome Órgão Superior,Código Órgão,Nome Órgão,Código Unidade Gestora,Nome Unidade Gestora,Ano e mês do lançamento,Valor Recebido
0,48891147000145,'' ASSOCIACAO DE PAIS E MESTRES DA ESCOLA ESTA...,SP,OSASCO,26000,Ministério da Educação,26298,Fundo Nacional de Desenvolvimento da Educação,153173,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,09/2026,4645
1,06343056000125,'' DROGARIA ECONOMICA DE VINHEDO LTDA,SP,VINHEDO,36000,Ministério da Saúde,36000,Ministério da Saúde - Unidades com vínculo direto,257001,DIRETORIA EXECUTIVA DO FUNDO NAC. DE SAUDE,09/2026,"48491,5300"


In [6]:
df.shape

(736840, 12)

In [15]:
df['Valor Recebido'] = (df['Valor Recebido']
                        .str.replace('.', '', regex=False)
                        .str.replace(',', '.', regex=False)
                        .astype(float))
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736840 entries, 0 to 736839
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Código Favorecido        736840 non-null  object 
 1   Nome Favorecido          736840 non-null  object 
 2   Sigla UF                 736840 non-null  object 
 3   Nome Município           736840 non-null  object 
 4   Código Órgão Superior    736840 non-null  int64  
 5   Nome Órgão Superior      736840 non-null  object 
 6   Código Órgão             736840 non-null  int64  
 7   Nome Órgão               736840 non-null  object 
 8   Código Unidade Gestora   736840 non-null  int64  
 9   Nome Unidade Gestora     736840 non-null  object 
 10  Ano e mês do lançamento  736840 non-null  object 
 11  Valor Recebido           736840 non-null  float64
dtypes: float64(1), int64(3), object(8)
memory usage: 67.5+ MB


In [ ]:
# Q1: Top 15 recipients by total amount received

# Group by 'Nome Favorecido' and sum
top_recipients = (df.groupby('Nome Favorecido')['Valor Recebido']
                  .sum()
                  .sort_values(ascending=False)
                  .head(15))

top_recipients

Nome Favorecido
PAGAMENTO DA DIVIDA INTERNA                             2.529064e+11
PAGAMENTO DE BENEFICIOS - INSS/FRGPS                    8.503565e+10
PAGAMENTOS DO TESOURO AO INSS                           5.707347e+10
COORDENACAO-GERAL DE TESOURARIA - CGTES                 4.683289e+10
FUNDOS DE PARTICIPACAO - FPM E FPE                      4.034647e+10
BANCO DO BRASIL SA                                      1.527496e+10
CAIXA ECONOMICA FEDERAL - AG.OPERADORA PBF              1.294000e+10
CAIXA ECONOMICA FEDERAL                                 1.255242e+10
FOLHA DE PAGAMENTO                                      1.234510e+10
BANCO NACIONAL DE DESENVOLVIMENTO ECONOMICO E SOCIAL    1.145858e+10
PAGAMENTOS DA SUBCONTA TESOURO PARA A SUBCONTA PI       1.096003e+10
MOVIMENTACOES DE CONCILICACAO DO SPB                    8.975282e+09
PETROLEO BRASILEIRO S A PETROBRAS                       6.398173e+09
SEM INFORMACAO                                          6.330973e+09
COORD-GERAL DE REC

In [26]:
# Q2: How many distinct recipients (`.nunique()`), split CPF vs CNPJ if the ID column allows. 

distinct_recipients = df['Nome Favorecido'].nunique()

distinct_recipients

623070

In [ ]:
# Verification of the lenght of the column 'Código Favorecido' → CPF vs CNPJ

df['Código Favorecido'].str.len().value_counts()

# It's not possible to split CPF vs CNPJ
# 2, 3, 6, 9, 10 and 14 lengths

Código Favorecido
14    728348
9       3858
2       2490
6       1252
3        889
10         3
Name: count, dtype: int64

In [32]:
# Q3: Recipients by number of separate payments (`value_counts()` sorted desc)

recipients_by_payments = (
    df['Nome Favorecido']
    .value_counts()
    .sort_values(ascending=False)
)

recipients_by_payments.head(30)

Nome Favorecido
SEM INFORMACAO                                                             2493
BANCO DO BRASIL SA                                                         2342
FUNDO MUNICIPAL DE ASSISTENCIA SOCIAL                                      2022
FUNDO MUNICIPAL DE SAUDE                                                   1891
FOLHA DE PAGAMENTO                                                          928
FUNDO MUNICIPAL DE ASSISTENCIA SOCIAL - FMAS                                667
PRIME CONSULTORIA E ASSESSORIA EMPRESARIAL LTDA                             602
CAIXA ECONOMICA FEDERAL                                                     476
SECRETARIA MUNICIPAL DE EDUCACAO                                            431
PAGAMENTOS PIX COM LISTA                                                    281
BANCO BRADESCO S.A.                                                         266
ITAU UNIBANCO S.A.                                                          263
BANCO SANTANDER (BRASIL)

In [ ]:
# VERIFICATIONS

# Summing up only positive number from 'Valor Recebido

total_positive_value = df[df['Valor Recebido'] > 0]['Valor Recebido'].sum()

# Summing up only negative number from 'Valor Recebido'

total_negative_value = df[df['Valor Recebido'] < 0]['Valor Recebido'].sum()

[total_positive_value, total_negative_value, total_positive_value + total_negative_value]

[np.float64(657440705131.76),
 np.float64(-566168894472.79),
 np.float64(91271810658.96997)]

In [37]:
# Top 3 negative recipients by total amount received

top_negative_recipients = (df[df['Valor Recebido'] < 0]
                          .groupby('Nome Favorecido')['Valor Recebido']
                            .sum()
                            .sort_values(ascending=True)
                            .head(3))

top_negative_recipients

Nome Favorecido
RECEBIMENTO DE EMISSAO DE TITULOS                 -2.820507e+11
RESGATE DE APLIC. DISPONIB. TN - ADM. DIV. PUBL   -2.264722e+11
PAGAMENTOS DO INSS AO TESOURO NACIONAL            -5.580225e+10
Name: Valor Recebido, dtype: float64